# 🫀 Risk Factors for Hypertension — Part 3B
## Multivariate Logistic Regression & Forest Plot
### Epidemiology with Python | Tutorial Series by Desy Nuryunarsih

---
Welcome back! In **Part 3A** we explored risk factors one at a time using  
chi-square tests and crude Odds Ratios.

**In Part 3B, you will learn to:**
- Run **multivariate logistic regression** to get **adjusted Odds Ratios (aOR)**
- Understand confounding and why adjustment matters
- Evaluate model performance (Accuracy, AUC, ROC curve)
- Compare crude vs adjusted ORs
- Build a **publication-quality forest plot**

> 💡 Run Part 3A first — we reuse the same simulated dataset here.


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, roc_curve)
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['font.family'] = 'sans-serif'

# ── Recreate the same simulated dataset from Part 3A ──────────────────
np.random.seed(2024)
n_case, n_ctrl = 250, 100
outcome = [1] * n_case + [0] * n_ctrl

def sim(p_case, p_ctrl, seed=0):
    np.random.seed(2024 + seed)
    return list(np.random.binomial(1, p_case, n_case)) +            list(np.random.binomial(1, p_ctrl,  n_ctrl))

data = {
    'hypertension':        outcome,
    'high_salt_diet':      sim(0.72, 0.35, 1),
    'physically_inactive': sim(0.65, 0.32, 2),
    'overweight_obese':    sim(0.60, 0.28, 3),
    'family_history':      sim(0.55, 0.25, 4),
    'high_stress':         sim(0.50, 0.30, 5),
    'smoker':              sim(0.42, 0.28, 6),
    'alcohol_use':         sim(0.30, 0.22, 7),
    'low_education':       sim(0.35, 0.30, 8),
    'fruit_veg_daily':     sim(0.25, 0.50, 9),
    'regular_checkup':     sim(0.20, 0.45, 10),
}

df = pd.DataFrame(data)
print("✅ Dataset recreated:", df.shape)
print("Cases:", df['hypertension'].sum(), "| Controls:", (df['hypertension']==0).sum())


## 📚 Step 1: Why Multivariate Logistic Regression?

In Part 3A, each risk factor was analysed **separately** (univariate = crude).  
But exposures are correlated — for example, people with high salt diets are  
often also physically inactive. This overlap is called **confounding**.

**Multivariate logistic regression** includes all variables at once and estimates  
each factor's **independent** contribution to hypertension risk.

### The model:
```
log( P(HTN) / (1 - P(HTN)) ) = β₀ + β₁X₁ + β₂X₂ + ... + βₙXₙ
```

**Adjusted OR = exp(β)**  
- aOR > 1 → independent risk factor for hypertension  
- aOR < 1 → independently protective  
- 95% CI excludes 1.0 → statistically significant

### Crude OR vs Adjusted OR:
| | Crude OR | Adjusted OR |
|---|---|---|
| Controls for confounders? | ❌ No | ✅ Yes |
| Looks at one variable? | ✅ Yes | ❌ No — all together |
| Best for? | Screening | Publication, inference |


In [ ]:
# ══════════════════════════════════════════════════════
# Step 2: Fit Multivariate Logistic Regression
# ══════════════════════════════════════════════════════

feature_cols = [
    'high_salt_diet', 'physically_inactive', 'overweight_obese',
    'family_history', 'high_stress', 'smoker', 'alcohol_use',
    'low_education', 'fruit_veg_daily', 'regular_checkup'
]

X = df[feature_cols]
y = df['hypertension']

# Use statsmodels for proper p-values and 95% CIs
X_sm = sm.add_constant(X)
logit_model = sm.Logit(y, X_sm)
result = logit_model.fit(disp=0)

print(result.summary2())


In [ ]:
# ══════════════════════════════════════════════════════
# Step 3: Extract Adjusted Odds Ratios & 95% CI
# ══════════════════════════════════════════════════════

coef     = result.params[1:]
pvalues  = result.pvalues[1:]
conf_int = result.conf_int().iloc[1:]

adj_or   = np.exp(coef)
ci_lower = np.exp(conf_int[0])
ci_upper = np.exp(conf_int[1])

lr_df = pd.DataFrame({
    'Variable':  feature_cols,
    'β (coef)':  coef.values.round(3),
    'Adj OR':    adj_or.values.round(2),
    'CI Lower':  ci_lower.values.round(2),
    'CI Upper':  ci_upper.values.round(2),
    'p-value':   pvalues.values.round(3),
    'Significant': pvalues.values < 0.05
})

lr_df['95% CI'] = lr_df.apply(
    lambda r: f"({r['CI Lower']} – {r['CI Upper']})", axis=1)
lr_df['Sig'] = lr_df['Significant'].map({True: '✅', False: '—'})

print("=== Adjusted Odds Ratios (Multivariate Logistic Regression) ===\n")
print(lr_df[['Variable', 'β (coef)', 'Adj OR', '95% CI', 'p-value', 'Sig']].to_string(index=False))


In [ ]:
# ══════════════════════════════════════════════════════
# Step 4: Model Performance
# ══════════════════════════════════════════════════════

# Use sklearn for easy metrics
from sklearn.linear_model import LogisticRegression
sk_model = LogisticRegression(random_state=42, max_iter=1000)
sk_model.fit(X, y)

y_pred       = sk_model.predict(X)
y_pred_proba = sk_model.predict_proba(X)[:, 1]
auc          = roc_auc_score(y, y_pred_proba)

print("=== Classification Report ===")
print(classification_report(y, y_pred,
      target_names=['Control (No HTN)', 'Hypertension']))
print(f"ROC AUC: {auc:.3f}")

# Confusion matrix + ROC curve side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['Predicted\nControl', 'Predicted\nHTN'],
            yticklabels=['Actual Control', 'Actual HTN'],
            annot_kws={"size": 14})
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y, y_pred_proba)
axes[1].plot(fpr, tpr, color='firebrick', lw=2.5,
             label=f'ROC curve (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=11)
axes[1].grid(True, linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('part3b_model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: part3b_model_performance.png")


## 🌲 Step 5: What is a Forest Plot?

A **forest plot** is the standard way to visualise Odds Ratios in epidemiology papers.

Each row shows one risk factor:
- **Square / dot** = point estimate (the OR)
- **Horizontal line** = 95% Confidence Interval
- **Vertical dashed line** at OR = 1.0 = the "line of no effect"

### Reading the forest plot:
- CI entirely to the **right** of 1.0 → significant **risk factor**
- CI entirely to the **left** of 1.0 → significant **protective factor**
- CI **crosses** 1.0 → result is **not statistically significant**


In [ ]:
# ══════════════════════════════════════════════════════
# Step 6: Publication-Quality Forest Plot
# ══════════════════════════════════════════════════════

plot_df = lr_df.sort_values('Adj OR').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 8))

for i, row in plot_df.iterrows():
    sig = row['Significant']
    color = ('firebrick'   if sig and row['Adj OR'] > 1 else
             'forestgreen' if sig and row['Adj OR'] < 1 else
             'steelblue')
    size = 130 if sig else 70

    # CI line
    ax.plot([row['CI Lower'], row['CI Upper']], [i, i],
            color=color, linewidth=2.5, alpha=0.85, zorder=2)
    # CI caps
    for x in [row['CI Lower'], row['CI Upper']]:
        ax.plot([x, x], [i - 0.13, i + 0.13],
                color=color, linewidth=2.5, alpha=0.85, zorder=2)
    # Point estimate
    ax.scatter(row['Adj OR'], i,
               color=color, s=size, edgecolors='black',
               linewidth=1.2, zorder=3)
    # Label
    annotation = (f"aOR = {row['Adj OR']:.2f} "
                  f"({row['CI Lower']:.2f}–{row['CI Upper']:.2f}), "
                  f"p = {row['p-value']:.3f}")
    ax.text(row['CI Upper'] + 0.12, i, annotation,
            va='center', ha='left', fontsize=9)

# Reference line
ax.axvline(x=1, color='black', linestyle='--', linewidth=1.8, zorder=1)

# Background shading
x_max = plot_df['CI Upper'].max() + 3
ax.axvspan(0.05, 1,     alpha=0.05, color='green')
ax.axvspan(1,    x_max, alpha=0.05, color='red')
ax.text(0.3,  -0.9, '← Protective',  fontsize=10, fontstyle='italic', color='forestgreen')
ax.text(1.1,  -0.9, 'Risk factor →', fontsize=10, fontstyle='italic', color='firebrick')

# Axes
var_labels = [r['Variable'].replace('_', ' ').title()
              for _, r in plot_df.iterrows()]
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(var_labels, fontsize=11)
ax.set_xlabel('Adjusted Odds Ratio (95% CI)', fontsize=12, fontweight='bold')
ax.set_xlim(0.05, x_max)
ax.set_title(
    'Forest Plot: Adjusted Odds Ratios for Hypertension Risk Factors\n'
    'Multivariate Logistic Regression, N=500 (250 cases, 250 controls)',
    fontsize=13, fontweight='bold', pad=15)

legend_elements = [
    mpatches.Patch(facecolor='firebrick',   label='Significant risk factor (aOR > 1, p < 0.05)'),
    mpatches.Patch(facecolor='forestgreen', label='Significant protective factor (aOR < 1, p < 0.05)'),
    mpatches.Patch(facecolor='steelblue',   label='Not significant (p ≥ 0.05)'),
]
ax.legend(handles=legend_elements, loc='lower right',
          fontsize=10, framealpha=0.92)
ax.grid(axis='x', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig('part3b_forest_plot.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Saved: part3b_forest_plot.png")


In [ ]:
# ══════════════════════════════════════════════════════
# Step 7: Compare Crude OR vs Adjusted OR
# ══════════════════════════════════════════════════════
# This reveals which associations were CONFOUNDED

def crude_or(df, exposure):
    t = pd.crosstab(df[exposure], df['hypertension'])
    a, b, c, d = t.loc[1,1], t.loc[1,0], t.loc[0,1], t.loc[0,0]
    if 0 in [a, b, c, d]:
        return float('nan')
    return round((a*d)/(b*c), 2)

rows = []
for col in feature_cols:
    cr = crude_or(df, col)
    adj_row = lr_df[lr_df['Variable'] == col].iloc[0]
    aOR = adj_row['Adj OR']
    if not np.isnan(cr) and cr != 0:
        pct_change = round((aOR - cr) / cr * 100, 1)
        note = ('↓ attenuated (confounded upward)'  if pct_change < -15 else
                '↑ amplified (confounded downward)'  if pct_change >  15 else
                '≈ stable (no major confounding)')
    else:
        pct_change, note = float('nan'), '—'

    rows.append({
        'Variable':       col.replace('_', ' ').title(),
        'Crude OR':       cr,
        'Adjusted OR':    aOR,
        '% Change':      f"{pct_change:+.1f}%" if not np.isnan(pct_change) else '—',
        'Confounding?':   note,
        'Adj p-value':    round(adj_row['p-value'], 3)
    })

comp_df = pd.DataFrame(rows)
print("=== Crude OR vs Adjusted OR ===\n")
print(comp_df.to_string(index=False))
print("\n💡 When adjusted OR differs from crude OR by >15%,")
print("   confounding is present and the adjusted estimate is preferred.")


## ✅ Part 3B Summary

| Concept | Key Point |
|---|---|
| Multivariate logistic regression | Controls for all variables simultaneously |
| Adjusted OR (aOR) | The independent effect of each exposure |
| Confounding | When crude OR ≠ adjusted OR substantially |
| Forest plot | Standard way to display ORs in publications |
| AUC | How well the model distinguishes cases from controls |
| Model accuracy | % of cases correctly classified |

---

### 🔍 Key findings (adjusted analysis):
- **High salt diet, physical inactivity, overweight, family history** remained **significant** after adjustment
- **Fruit & vegetable intake, regular check-up** remained **protective**
- Some variables attenuated after adjustment → confounding was present

---

### 🎓 Series recap — what you've learned so far:
| Video | Topic |
|---|---|
| Video 1 | Introduction to Epidemiology & Python |
| Video 2 | Measures of Association (RR, RD, OR, NNT, IRR, IRD) |
| Video 3A | Case-control design, chi-square, crude OR |
| Video 3B | Logistic regression, adjusted OR, forest plot |

**Next up:** Confounding & Effect Modification — subscribe so you don't miss it! 🔔

---
*Tutorial by Desy Nuryunarsih | Research Fellow, University of St Andrews*  
*YouTube: Epidemiology with Python*
